# Просмотр готового графа GraphRAG
Этот ноутбук читает готовые результаты. Ollama и повторная индексация не нужны.

Запуск: `uv run jupyter lab graph-viewer.ipynb`, ядро **Python (graph-hw)**.
Выполните ячейки сверху вниз. По умолчанию выбирается последний рассчитанный граф.
Узлы можно перетаскивать, масштабировать колесом и выбирать щелчком; щелчок по линии показывает описание связи.
Цвет обозначает тип сущности, размер узла — число связей в отображаемом графе.
Линии показаны без стрелок: порядок source/target сам по себе не доказывает направление причинности.
Описания получены языковой моделью и требуют проверки по исходному тексту.


In [1]:
from pathlib import Path
import json, html, math, sys
import pandas as pd
from pyvis.network import Network
from IPython.display import HTML, display

ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent) if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "Запустите Jupyter из папки graph_hw или hw2"
sys.path.insert(0, str(ROOT))
runs = sorted(
    [p.parent for p in (ROOT / "local_runs").glob("*/output/artifacts/entities.parquet")
     if (p.parent / "relationships.parquet").exists()],
    key=lambda p: (p / "entities.parquet").stat().st_mtime, reverse=True,
)
if not runs:
    raise FileNotFoundError("Нет результатов. Сначала завершите индексацию в локальном ноутбуке.")
for i, p in enumerate(runs):
    print(f"[{i}] {p.parent.parent.name}")


[0] ganoshenko-full-294d037a3646
[1] vanadiy_review-full-1c7c65ac3aae
[2] vanadiy_review-full-b0ce1852c3bd
[3] vanadiy_review-full-0fea01adef45
[4] vanadiy_review-sample-6b216c568da9
[5] vanadiy_review-sample-8191ea1eaf2c
[6] vanadiy_review-sample-a36d193aff55


## Выбор графа
Измените `RUN_INDEX`, если доступно несколько запусков. Фильтры необязательны. `MAX_NODES = None` показывает все узлы; ограничение пригодится для большого полного графа.
По умолчанию используется нормализованное представление, скрыты изолированные узлы и узлы с пометкой «проверить». Их можно вернуть флагами HIDE_ISOLATED и HIDE_REVIEW_NODES. Это фильтры отображения, а не удаление знаний. USE_NORMALIZED=False возвращает исходный граф. Старым графам новые цитаты и связи не добавляются.

In [2]:
RUN_INDEX = 0
USE_NORMALIZED = True
HIDE_ISOLATED = True
HIDE_REVIEW_NODES = True  # Подозрительные и внепредметные узлы остаются в аудите
MAX_NODES = None  # Например, 300 для большого графа
ENTITY_TYPES = []  # Пусто — все типы; например ["ХИМИЧЕСКИЙ_ЭЛЕМЕНТ", "МАТЕРИАЛ"]

ARTIFACTS = runs[RUN_INDEX]
DATA_DIR = ARTIFACTS
if USE_NORMALIZED:
    from graph_quality import normalize_graph
    DATA_DIR, quality = normalize_graph(ARTIFACTS)
    print("Нормализация:", quality)
entities = pd.read_parquet(DATA_DIR / "entities.parquet")
relationships = pd.read_parquet(DATA_DIR / "relationships.parquet")
assert {"title", "type", "description"} <= set(entities.columns)
assert {"source", "target", "description"} <= set(relationships.columns)
if entities["title"].duplicated().any():
    raise ValueError("Есть повторяющиеся названия узлов; нужно проверить таблицу entities.")
selected = entities.copy()
if HIDE_REVIEW_NODES and "review_reason" in selected:
    selected = selected[selected["review_reason"].eq("")]
if ENTITY_TYPES:
    selected = selected[selected["type"].isin(ENTITY_TYPES)]
if MAX_NODES is not None:
    if MAX_NODES < 1:
        raise ValueError("MAX_NODES должен быть положительным или None")
    selected = selected.sort_values("degree", ascending=False).head(MAX_NODES)
names = set(selected["title"])
edges = relationships[relationships["source"].isin(names) & relationships["target"].isin(names)].copy()
if HIDE_ISOLATED:
    connected = set(edges["source"]) | set(edges["target"])
    selected = selected[selected["title"].isin(connected)]
if selected.empty:
    raise ValueError("Фильтр не оставил узлов")
print("Запуск:", ARTIFACTS.parent.parent.name)
print(f"Показано {len(selected)} из {len(entities)} узлов, {len(edges)} из {len(relationships)} связей")
display(selected.groupby("type", dropna=False).size().rename("Узлов").to_frame())


Нормализация: {'version': 'metallurgy-v3', 'raw_nodes': 606, 'nodes': 606, 'raw_edges': 1123, 'edges': 1123, 'isolated_nodes': 91, 'nodes_to_review': 19, 'edges_without_quotes': 6, 'edges_with_unmatched_quotes': 237, 'warning': 'Совпадение цитаты не подтверждает смысл связи; отсутствие флагов не означает отсутствие ошибок.'}
Запуск: ganoshenko-full-294d037a3646
Показано 502 из 606 узлов, 1112 из 1123 связей


,Узлов
type,
МАТЕРИАЛ,88
МИКРОСТРУКТУРА,150
СВОЙСТВО,76
СОЕДИНЕНИЕ,41
ТЕХНОЛОГИЧЕСКИЙ_ПРОЦЕСС,120
ХИМИЧЕСКИЙ_ЭЛЕМЕНТ,27


## Интерактивный граф
Поиск находит узлы по части названия. После выбора узла справа видны описание и его соседи. Кнопка «Уместить граф» возвращает общий вид. Если Jupyter скрывает визуализацию, выполните ячейку заново или откройте сохранённый `graph.html` в браузере.
Выбор узла выделяет его соседей и показывает краткие подписи прилегающих связей (начало описания, не отдельная классификация отношений). «Уместить граф» сбрасывает выделение.

In [3]:
palette = ["#2563eb", "#16a34a", "#ea580c", "#9333ea", "#0891b2", "#dc2626", "#a16207", "#64748b"]
types = sorted(selected["type"].fillna("НЕИЗВЕСТНО").unique())
colors = {t: palette[i % len(palette)] for i, t in enumerate(types)}
degree = pd.concat([edges["source"], edges["target"]]).value_counts()
net = Network(height="650px", width="100%", bgcolor="#ffffff", font_color="#172033", cdn_resources="in_line", directed=False)
node_details, edge_details = {}, {}
for row in selected.fillna({"description": "", "type": "НЕИЗВЕСТНО"}).to_dict("records"):
    name = str(row["title"])
    n = int(degree.get(name, 0))
    net.add_node(name, label=name, color=colors[row["type"]], size=12 + 3 * math.sqrt(n))
    node_details[name] = {"type": row["type"], "description": str(row["description"]), "degree": n}
for i, row in enumerate(edges.fillna({"description": ""}).to_dict("records")):
    edge_id = f"edge-{i}"
    net.add_edge(str(row["source"]), str(row["target"]), id=edge_id, width=1.5, color="#94a3b8")
    edge_details[edge_id] = {"source": str(row["source"]), "target": str(row["target"]), "description": str(row["description"])}
net.set_options(json.dumps({
    "layout": {"randomSeed": 42, "improvedLayout": True},
    "nodes": {"shape": "dot", "font": {"size": 13}},
    "edges": {"smooth": {"type": "continuous"}},
    "interaction": {"hover": True, "navigationButtons": True, "keyboard": True},
    "physics": {"solver": "forceAtlas2Based", "stabilization": {"iterations": 250},
                "forceAtlas2Based": {"gravitationalConstant": -70, "springLength": 150}},
}))
page = net.generate_html(notebook=False)
import re
page = re.sub(r'<script\b[^>]*\bsrc=[^>]*>.*?</script>', '', page, flags=re.DOTALL)
page = re.sub(r'<link\b[^>]*\bhref=[^>]*>', '', page, flags=re.DOTALL)
legend = " ".join(f'<span style="color:{colors[t]};margin-right:16px">● {html.escape(t)}</span>' for t in types)
toolbar = f"""<section id="viewer-toolbar" style="padding:16px;font-family:Arial,sans-serif">
<h2 style="margin:0 0 12px">{html.escape(ARTIFACTS.parent.parent.name)}</h2>
<p>{len(selected)} узлов · {len(edges)} связей</p>
<label>Найти узел <input id="node-search" type="search" placeholder="Например, ниобий"></label>
<label>Выбрать <select id="node-select"><option value="">Все узлы</option></select></label>
<button id="fit-graph" type="button">Уместить граф</button>
<button id="toggle-physics" type="button">Остановить движение</button>
<div style="margin-top:12px;line-height:1.8">{legend}</div></section>
<aside id="graph-detail" style="padding:16px;font:14px/1.6 Arial,sans-serif;white-space:pre-wrap;overflow-wrap:anywhere" aria-live="polite">Выберите узел или связь.</aside>
"""
page = page.replace("<body>", "<body>" + toolbar, 1)
def script_json(value):
    return json.dumps(value, ensure_ascii=False).replace("<", chr(92)+"u003c").replace(">", chr(92)+"u003e").replace("&", chr(92)+"u0026")
script = r"""
<style>
#viewer-toolbar input,#viewer-toolbar select,#viewer-toolbar button {padding:8px;margin:4px;max-width:95%}
#mynetwork {border:1px solid #cbd5e1!important; float:none!important; width:100%!important}
#graph-detail {max-height:340px;overflow:auto;background:#f8fafc}
</style>
<script>
const nodeDetails = NODE_DATA;
const edgeDetails = EDGE_DATA;
const picker = document.getElementById('node-select');
const search = document.getElementById('node-search');
const detail = document.getElementById('graph-detail');
const names = Object.keys(nodeDetails).sort((a,b)=>a.localeCompare(b,'ru'));
function fillPicker(query='') {
 picker.replaceChildren(new Option('Все узлы',''));
 names.filter(n=>n.toLocaleLowerCase().includes(query.toLocaleLowerCase())).forEach(n=>picker.add(new Option(n,n)));
}
function describeNode(id) {
 const d=nodeDetails[id];
 if(!d) return;
 const connected=new Set([id,...network.getConnectedNodes(id)]);
 nodes.update(nodes.get().map(n=>({id:n.id,opacity:connected.has(n.id)?1:0.15})));
 edges.update(edges.get().map(e=>({id:e.id,label:(e.from===id||e.to===id)?edgeDetails[e.id].description.split('Цитата:')[0].slice(0,65):''})));
 detail.textContent=id+'\nТип: '+d.type+' · Связей: '+d.degree+'\n\n'+d.description+'\n\nСоседи: '+network.getConnectedNodes(id).join(', ');
}
fillPicker();
search.addEventListener('input',()=>fillPicker(search.value));
picker.addEventListener('change',()=>{
 if(!picker.value) {network.unselectAll();network.fit();return;}
 network.selectNodes([picker.value]);network.focus(picker.value,{scale:1.1,animation:true});describeNode(picker.value);
});
network.on('click',p=>{
 if(p.nodes.length) describeNode(p.nodes[0]);
 else if(p.edges.length) {const e=edgeDetails[p.edges[0]];if(e) detail.textContent=e.source+' — '+e.target+'\n\n'+e.description;}
});
document.getElementById('fit-graph').onclick=()=>{network.unselectAll();nodes.update(nodes.get().map(n=>({id:n.id,opacity:1})));edges.update(edges.get().map(e=>({id:e.id,label:''})));network.fit({animation:true});};
let moving=true;
document.getElementById('toggle-physics').onclick=()=>{
 moving=!moving;network.setOptions({physics:{enabled:moving}});
 document.getElementById('toggle-physics').textContent=moving?'Остановить движение':'Включить движение';
};
</script>
"""
script = script.replace("NODE_DATA", script_json(node_details)).replace("EDGE_DATA", script_json(edge_details))
page = page.replace("</body>", script + "</body>")
HTML_PATH = ARTIFACTS.parent.parent / "graph.html"
HTML_PATH.write_text(page, encoding="utf-8")
display(HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" width="100%" height="1120" style="border:0" sandbox="allow-scripts allow-same-origin"></iframe>'))
print("Открыть отдельно в браузере:", HTML_PATH)

c:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\.venv\Lib\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Открыть отдельно в браузере: C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\local_runs\ganoshenko-full-294d037a3646\graph.html


## Проверка по исходному фрагменту
Введите часть названия. Ниже выводятся связанные фрагменты входного текста, если сохранился `text_units.parquet`. Это помогает проверять утверждения модели.

In [4]:
QUERY = "НИОБИЙ"
matches = entities[entities["title"].str.contains(QUERY, case=False, regex=False, na=False)]
display(matches[["title", "type", "description"]])
units_path = ARTIFACTS / "text_units.parquet"
if units_path.exists() and "text_unit_ids" in matches:
    units = pd.read_parquet(units_path)
    ids = {str(uid) for values in matches["text_unit_ids"] for uid in (values if values is not None else [])}
    for row in units[units["id"].isin(ids)].to_dict("records"):
        print("Фрагмент:", row["id"])
        print(row["text"])
        print()


,title,type,description
287,НИОБИЙ,ХИМИЧЕСКИЙ_ЭЛЕМЕНТ,"Ниобий — это химический элемент, используемый ..."


Фрагмент: eb42c59431492dd7fb44b12bad06e8a619dc5c39eb32a8397e38fc4b53d1242e96fd650815420e0babd11244656904a29eb5cd62a39ff497dc1fe17809d49cdf
. Литвиненко,

## ВВЕДЕНИЕ

С.А. Голованенко, М.Л. Бернштейна, Н.П. Лякишева, П.Д. Одесского,
В.Н. Зикеева, Л.И. Эфрона, Ю.Д. Морозова и др. ученых.

Переход от сталей категорий прочности К60 (X70) к сталям категорий прочности К65 (X80) и более прочным требует пересмотра металловедческий принципов их легирования и микролегирования и новых технологических решений. Получение уровня прочности $\sigma_{\mathrm{B}} \geq 630 \mathrm{~H} / \mathrm{mm}^{2}$, $\sigma_{\mathrm{T}} \geq 570 \mathrm{~H} / \mathrm{mm}^{2}$ в листах в сочетании с другими важнейшими показателями механических свойств ($\delta_{5} \geq 22\%$; KCV при -20 °C ≥ 130 Дж/см²; доли вязкой составляющей в изломах образцов ИПГ ≥ 95 % при -20 °C) становится невозможным на базе ферритно-перлитной структуры и требует перехода к иному структурному состоянию материала — к сталям с дисперсной ферр